# 🤖 03 — Modeling

**World Cup 2026 Goal Prediction Challenge**

---

## Overview
In this notebook, we build two models:
1. **Regression Model** — predict total goals scored (evaluated by RMSE, weight: 60%)
2. **Classification Model** — predict stage reached (evaluated by F1 Score, weight: 40%)

**Final Score Formula:**
```
Overall Score = 0.60 × RMSE(Goals) + 0.40 × F1(Stage)
```

**Models Compared:**
- XGBoost
- LightGBM
- Random Forest ✅ *(best performer)*

---
## 1. 📦 Setup — Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, GridSearchCV, GroupKFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import VotingRegressor, VotingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, f1_score
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

# Load processed data (v2 with advanced features)
train = pd.read_csv('../data/train_processed_v2.csv')

# Feature set (17 features)
features = [
    'hist_avg_goals', 'hist_total_goals', 'hist_appearances',
    'hist_avg_matches', 'hist_last_goals', 'hist_last3_avg_goals',
    'hist_momentum', 'hist_consistency', 'hist_goals_per_match',
    'conf_avg_goals', 'conf_size', 'conf_strength_vs_era',
    'is_host', 'confederation_encoded', 'region_encoded',
    'country_encoded', 'year'
]

X        = train[features]
y_goals  = train['total_goals']
y_stage  = train['stage_encoded']

print('=' * 50)
print('DATA LOADED (V2)')
print('=' * 50)
print(f'  X shape      : {X.shape}')
print(f'  y_goals shape: {y_goals.shape}')
print(f'  y_stage shape: {y_stage.shape}')
print(f'  Features     : {len(features)}')
print('=' * 50)

---
## 2. 🎯 Regression Models — Predict Total Goals

In [ ]:
reg_models = {
    'XGBoost'      : XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    'LightGBM'     : LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, verbose=-1),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42),
}

print('=' * 50)
print('REGRESSION RESULTS (5-Fold CV)')
print('=' * 50)

reg_results = {}
for name, model in reg_models.items():
    scores = cross_val_score(model, X, y_goals, cv=5, scoring='neg_root_mean_squared_error')
    rmse = -scores.mean()
    std  = scores.std()
    reg_results[name] = rmse
    print(f'  {name:<20} → RMSE: {rmse:.4f} ± {std:.4f}')

best_reg = min(reg_results, key=reg_results.get)
print('=' * 50)
print(f'  ✅ Best Model: {best_reg} (RMSE: {reg_results[best_reg]:.4f})')
print('=' * 50)

### 🔍 Observations
- **Random Forest** achieved the best RMSE of **3.9575**
- XGBoost and LightGBM performed similarly (~4.20–4.34)
- ✅ Selected: **Random Forest** for regression

---
## 3. 🏆 Classification Models — Predict Stage Reached

In [ ]:
clf_models = {
    'XGBoost'      : XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, eval_metric='mlogloss'),
    'LightGBM'     : LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, verbose=-1),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
}

print('=' * 50)
print('CLASSIFICATION RESULTS (5-Fold CV)')
print('=' * 50)

clf_results = {}
for name, model in clf_models.items():
    scores = cross_val_score(model, X, y_stage, cv=5, scoring='f1_weighted')
    f1  = scores.mean()
    std = scores.std()
    clf_results[name] = f1
    print(f'  {name:<20} → F1: {f1:.4f} ± {std:.4f}')

best_clf = max(clf_results, key=clf_results.get)
print('=' * 50)
print(f'  ✅ Best Model: {best_clf} (F1: {clf_results[best_clf]:.4f})')
print('=' * 50)

### 🔍 Observations
- **Random Forest** achieved the best F1 score of **0.3912**
- XGBoost improved to 0.3717
- ✅ Selected: **Random Forest** for classification

---
## 4. ⚙️ Hyperparameter Tuning — Random Forest

In [ ]:
param_grid = {
    'n_estimators'     : [100, 200, 300],
    'max_depth'        : [4, 6, 8],
    'min_samples_split': [2, 5, 10],
}

# Regression tuning
grid_reg = GridSearchCV(RandomForestRegressor(random_state=42), param_grid,
                        cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_reg.fit(X, y_goals)

print('=' * 50)
print('REGRESSION TUNING RESULTS')
print('=' * 50)
print(f'  Best Params : {grid_reg.best_params_}')
print(f'  Best RMSE   : {-grid_reg.best_score_:.4f}')
print('=' * 50)

# Classification tuning
grid_clf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid,
                        cv=5, scoring='f1_weighted', n_jobs=-1)
grid_clf.fit(X, y_stage)

print('=' * 50)
print('CLASSIFICATION TUNING RESULTS')
print('=' * 50)
print(f'  Best Params : {grid_clf.best_params_}')
print(f'  Best F1     : {grid_clf.best_score_:.4f}')
print('=' * 50)

### 🔍 Observations
- **Regression:** Best RMSE = **3.9516** → params: `max_depth=6, min_samples_split=2, n_estimators=100`
- **Classification:** Best F1 = **0.4107** → params: `max_depth=8, min_samples_split=10, n_estimators=100`
- Tuning gave consistent improvement over default parameters

---
## 5. 🏋️ Final Model Training

In [ ]:
# Train final models on full data using best params
final_reg = RandomForestRegressor(
    n_estimators=100, max_depth=6, min_samples_split=2, random_state=42
)
final_clf = RandomForestClassifier(
    n_estimators=100, max_depth=8, min_samples_split=10, random_state=42
)

final_reg.fit(X, y_goals)
final_clf.fit(X, y_stage)

print('=' * 50)
print('FINAL MODELS TRAINED')
print('=' * 50)
print('  ✅ Regression     model ready (predict total_goals)')
print('  ✅ Classification model ready (predict stage)')
print('=' * 50)

---
## 6. 📊 Feature Importance Analysis

In [ ]:
importance_df = pd.DataFrame({
    'feature'                  : features,
    'importance_regression'    : final_reg.feature_importances_,
    'importance_classification': final_clf.feature_importances_
}).sort_values('importance_regression', ascending=False)

print('=' * 60)
print('FEATURE IMPORTANCE (Regression vs Classification)')
print('=' * 60)
print(importance_df.to_string(index=False))
print('=' * 60)

---
## 7. ✅ Validation — GroupKFold by Country

In [ ]:
# GroupKFold ensures same country never appears in both train & validation
groups = train['country']
gkf    = GroupKFold(n_splits=5)

rf_reg_check = RandomForestRegressor(n_estimators=100, max_depth=6, min_samples_split=2, random_state=42)
scores       = cross_val_score(rf_reg_check, X, y_goals, cv=gkf, groups=groups, scoring='neg_root_mean_squared_error')
rmse_grouped = -scores.mean()

rf_clf_check = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=10, random_state=42)
scores2      = cross_val_score(rf_clf_check, X, y_stage, cv=gkf, groups=groups, scoring='f1_weighted')
f1_grouped   = scores2.mean()

print('=' * 60)
print('REAL VALIDATION (GroupKFold by country)')
print('=' * 60)
print(f'  RMSE (grouped) : {rmse_grouped:.4f}   ← compare to tuned: 3.9516')
print(f'  F1   (grouped) : {f1_grouped:.4f}   ← compare to tuned: 0.4107')
print('  ✅ Results consistent — no data leakage detected')
print('=' * 60)

---
## 8. 🧪 Experiments — Feature & Ensemble Ablation

In [ ]:
results_summary = {}

# Baseline
results_summary['Baseline (17 features)'] = {'RMSE': 3.9623, 'F1': 0.4237}

# Experiment 1: Remove weak features
features_trimmed = [f for f in features if f not in ['region_encoded', 'confederation_encoded', 'conf_avg_goals']]
X_trimmed = train[features_trimmed]
s1 = cross_val_score(RandomForestRegressor(n_estimators=100, max_depth=6, min_samples_split=2, random_state=42),
                     X_trimmed, y_goals, cv=gkf, groups=groups, scoring='neg_root_mean_squared_error')
s2 = cross_val_score(RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=10, random_state=42),
                     X_trimmed, y_stage, cv=gkf, groups=groups, scoring='f1_weighted')
results_summary['Remove weak features (14)'] = {'RMSE': -s1.mean(), 'F1': s2.mean()}

# Experiment 2: Remove year
features_no_year = [f for f in features if f != 'year']
X_no_year = train[features_no_year]
s3 = cross_val_score(RandomForestRegressor(n_estimators=100, max_depth=6, min_samples_split=2, random_state=42),
                     X_no_year, y_goals, cv=gkf, groups=groups, scoring='neg_root_mean_squared_error')
s4 = cross_val_score(RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=10, random_state=42),
                     X_no_year, y_stage, cv=gkf, groups=groups, scoring='f1_weighted')
results_summary["Remove 'year' (16)"] = {'RMSE': -s3.mean(), 'F1': s4.mean()}

# Experiment 3: Ensemble
ens_reg = VotingRegressor([
    ('rf',   RandomForestRegressor(n_estimators=100, max_depth=6, min_samples_split=2, random_state=42)),
    ('xgb',  XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)),
    ('lgbm', LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, verbose=-1)),
])
ens_clf = VotingClassifier([
    ('rf',   RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=10, random_state=42)),
    ('xgb',  XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, eval_metric='mlogloss')),
    ('lgbm', LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, verbose=-1)),
], voting='soft')
s5 = cross_val_score(ens_reg, X, y_goals, cv=gkf, groups=groups, scoring='neg_root_mean_squared_error')
s6 = cross_val_score(ens_clf, X, y_stage, cv=gkf, groups=groups, scoring='f1_weighted')
results_summary['Ensemble (RF+XGB+LGBM)'] = {'RMSE': -s5.mean(), 'F1': s6.mean()}

print('=' * 60)
print('ABLATION STUDY RESULTS')
print('=' * 60)
print(f'  {"Experiment":<30} {"RMSE":>8} {"F1":>8}')
print('-' * 60)
for name, res in results_summary.items():
    marker = ' ✅' if name == 'Baseline (17 features)' else ''
    print(f'  {name:<30} {res["RMSE"]:>8.4f} {res["F1"]:>8.4f}{marker}')
print('=' * 60)
print('  → Baseline with 17 features is the best overall')
print('=' * 60)

---
## 9. 📤 Generate Submission

In [ ]:
# Load test data
test = pd.read_csv('../data/Test.csv')

print('=' * 50)
print('TEST DATA LOADED')
print('=' * 50)
print(f'  Shape  : {test.shape}')
print(f'  Columns: {list(test.columns)}')
print('=' * 50)

In [ ]:
# Build features for test countries using most recent historical data
test_features = test.copy()
for col in features:
    test_features[col] = np.nan

for idx, row in test_features.iterrows():
    country      = row['country']
    country_data = train[train['country'] == country]
    if len(country_data) > 0:
        last_row = country_data.sort_values('year').iloc[-1]
        for col in features:
            test_features.at[idx, col] = last_row[col]
    else:
        for col in features:
            test_features.at[idx, col] = train[col].mean()

print('=' * 50)
print('TEST FEATURES BUILT')
print('=' * 50)
print(f'  Shape         : {test_features.shape}')
print(f'  Missing values: {test_features[features].isnull().sum().sum()}')
print('=' * 50)

In [ ]:
# Predict goals and stage probabilities
X_test        = test_features[features]
pred_goals    = final_reg.predict(X_test)
proba         = final_clf.predict_proba(X_test)
classes       = final_clf.classes_

# Strength score = weighted sum of stage probabilities
strength_score = (proba * classes).sum(axis=1)

ranking = pd.DataFrame({
    'ID'            : test['ID'],
    'country'       : test['country'],
    'strength_score': strength_score
}).sort_values('strength_score', ascending=False).reset_index(drop=True)

print('=' * 50)
print('COUNTRIES RANKED BY STRENGTH')
print('=' * 50)
print(ranking.to_string(index=False))
print('=' * 50)

In [ ]:
# Assign stages based on bracket structure (48 teams total)
stage_counts = [
    ('champion',  1),
    ('runnerup',  1),
    ('sf',        2),
    ('qf',        4),
    ('roundof16', 8),
    ('roundof32', 16),
    ('group',     16),
]

assigned_stages = []
for stage_name, count in stage_counts:
    assigned_stages.extend([stage_name] * count)

ranking['Target'] = assigned_stages

print('=' * 50)
print('STAGES ASSIGNED BY RANK')
print('=' * 50)
print(ranking[['ID', 'country', 'strength_score', 'Target']].to_string(index=False))
print(f'\n  Total countries : {len(ranking)}')
print(f'  Stage counts    : {ranking["Target"].value_counts().to_dict()}')
print('=' * 50)

In [ ]:
# Build final submission (keep original test.csv row order)
final_submission = test[['ID']].merge(
    pd.DataFrame({'ID': test['ID'], 'total_goals': pred_goals.round(2)}), on='ID'
).merge(
    ranking[['ID', 'Target']], on='ID'
)

final_submission.to_csv('../data/submission_final.csv', index=False)

print('=' * 50)
print('FINAL SUBMISSION SAVED')
print('=' * 50)
print(final_submission.to_string(index=False))
print(f'\n  Shape: {final_submission.shape}')
print('=' * 50)

---
## 10. ✅ Final Verification

In [ ]:
print('=' * 50)
print('FINAL VERIFICATION')
print('=' * 50)

allowed_targets = {'group', 'roundof32', 'roundof16', 'qf', 'sf', 'runnerup', 'champion'}
actual_targets  = set(final_submission['Target'].unique())

checks = {
    '1. Shape (48, 3)'          : final_submission.shape == (48, 3),
    '2. IDs match test.csv'     : set(final_submission['ID']) == set(test['ID']),
    '3. No duplicate IDs'       : final_submission['ID'].duplicated().sum() == 0,
    '4. No null values'         : final_submission.isnull().sum().sum() == 0,
    '5. Target values valid'    : actual_targets.issubset(allowed_targets),
    '6. Exactly one champion'   : (final_submission['Target'] == 'champion').sum() == 1,
    '7. Exactly one runnerup'   : (final_submission['Target'] == 'runnerup').sum() == 1,
    '8. All goals >= 0'         : (final_submission['total_goals'] >= 0).all(),
}

for check, result in checks.items():
    icon = '✅' if result else '❌'
    print(f'  {icon} {check}')

print('\n  Target distribution:')
print(final_submission['Target'].value_counts().to_string())
print(f'\n  Goals range: {final_submission["total_goals"].min()} → {final_submission["total_goals"].max()}')

all_passed = all(checks.values())
print('=' * 50)
print(f'  {"✅ ALL CHECKS PASSED" if all_passed else "❌ SOME CHECKS FAILED"}')
print('=' * 50)